In [1]:
import numpy as np

def gram_schmidt(V):
    """
    Performs the Modified Gram-Schmidt orthogonalization process
    on a set of vectors.

    Args:
        V (np.ndarray): A 2D array where each *column* is a vector.
                        Shape: (M, N) where M is dimension, N is num_vectors.

    Returns:
        np.ndarray: A 2D array (Q) of the same shape, where each *column*
                    is an orthonormal vector.

    Raises:
        ValueError: If the vectors are linearly dependent (i.e., one of
                    the orthogonalized vectors becomes a zero vector).
    """

    # 1. Initialization
    # Work on a copy of the matrix, and ensure it's float type
    # for numerical precision during division.
    V_copy = V.astype(float).copy()
    num_rows, num_cols = V_copy.shape

    # Create an empty matrix Q to store the resulting orthonormal vectors
    Q = np.zeros_like(V_copy)

    # 2. Main Loop
    # Iterate through each column (vector) of the input matrix
    for i in range(num_cols):

        # 3. Get the current vector
        # v_i is the vector we are currently orthogonalizing.
        # Note: In iterations > 0, this v_i has already been
        # modified by the inner loop in previous steps.
        v_i = V_copy[:, i]

        # 4. Calculate the norm
        norm_v_i = np.linalg.norm(v_i)

        # 5. Check for Linear Dependence
        # If the norm is (close to) zero, it means this vector
        # is a linear combination of the previous vectors.
        if norm_v_i < 1e-9:
            raise ValueError(f"Vectors are linearly dependent. "
                             f"Failed at vector index {i}.")

        # 6. Normalize (create the i-th orthonormal vector)
        # This is our new basis vector, q_i
        q_i = v_i / norm_v_i
        Q[:, i] = q_i

        # 7. Modify Subsequent Vectors (The "Modified" part)
        # Now, subtract the projection of q_i from all *future* vectors.
        # This makes all subsequent vectors orthogonal to q_i.
        for j in range(i + 1, num_cols):
            # Get the j-th vector (which is still in V_copy)
            v_j = V_copy[:, j]

            # Find the projection coefficient: (v_j . q_i)
            # The full projection is (v_j . q_i) * q_i, since ||q_i|| = 1.
            projection_coeff = np.dot(v_j, q_i)

            # Subtract the projection from v_j
            # v_j = v_j - (projection_coeff * q_i)
            v_j_orthogonal = v_j - projection_coeff * q_i

            # Update the vector in V_copy for the next iteration
            V_copy[:, j] = v_j_orthogonal

    # 8. Return the orthonormal matrix Q
    return Q

if __name__ == "__main__":
    # Set print options for numpy for cleaner output
    np.set_printoptions(precision=4, suppress=True)

    # --- Example 1: A 3x3 linearly independent set ---
    # Vectors are [1, 1, 0], [1, 0, 1], [0, 1, 1]
    A = np.array([
        [1, 1, 0],
        [1, 0, 1],
        [0, 1, 1]
    ], dtype=float)

    print("--- Example 1 (3x3) ---")
    print("Original Matrix V (vectors as columns):\n", A)

    try:
        Q = gram_schmidt(A)
        print("\nOrthonormal Matrix Q:\n", Q)

        # Check: Q.T @ Q should be the Identity matrix
        # (The @ operator is for matrix multiplication)
        print("\nCheck (Q.T @ Q should be Identity):\n", Q.T @ Q)

    except ValueError as e:
        print("\nFailed:", e)

    print("-" * 30)

    # --- Example 2: A 3x2 set (non-square) ---
    B = np.array([
        [1, 2],
        [0, 3],
        [2, 1]
    ], dtype=float)

    print("\n--- Example 2 (3x2) ---")
    print("Original Matrix V:\n", B)

    try:
        Q_B = gram_schmidt(B)
        print("\nOrthonormal Matrix Q:\n", Q_B)

        # Check:
        print("\nCheck (Q.T @ Q should be Identity):\n", Q_B.T @ Q_B)

    except ValueError as e:
        print("\nFailed:", e)

    print("-" * 30)

    # --- Example 3: A linearly *dependent* set ---
    # Vector 2 ([2, 2]) is just 2 * Vector 1 ([1, 1])
    C = np.array([
        [1, 2],
        [1, 2]
    ], dtype=float)

    print("\n--- Example 3 (Linearly Dependent) ---")
    print("Original Matrix V:\n", C)

    try:
        Q_C = gram_schmidt(C)
        print("\nOrthonormal Matrix Q:\n", Q_C)
    except ValueError as e:
        print("\nFailed as expected:")
        print(e)

    print("-" * 30)

--- Example 1 (3x3) ---
Original Matrix V (vectors as columns):
 [[1. 1. 0.]
 [1. 0. 1.]
 [0. 1. 1.]]

Orthonormal Matrix Q:
 [[ 0.7071  0.4082 -0.5774]
 [ 0.7071 -0.4082  0.5774]
 [ 0.      0.8165  0.5774]]

Check (Q.T @ Q should be Identity):
 [[ 1.  0.  0.]
 [ 0.  1. -0.]
 [ 0. -0.  1.]]
------------------------------

--- Example 2 (3x2) ---
Original Matrix V:
 [[1. 2.]
 [0. 3.]
 [2. 1.]]

Orthonormal Matrix Q:
 [[ 0.4472  0.3651]
 [ 0.      0.9129]
 [ 0.8944 -0.1826]]

Check (Q.T @ Q should be Identity):
 [[1. 0.]
 [0. 1.]]
------------------------------

--- Example 3 (Linearly Dependent) ---
Original Matrix V:
 [[1. 2.]
 [1. 2.]]

Failed as expected:
Vectors are linearly dependent. Failed at vector index 1.
------------------------------
